# I_GEN Weight Calibration

Calibrates the w1-w10 weights in the ZetaScript control law using
the heatmap_data corpus from Sajadi et al. (2022).

Outputs physics-grounded weights to replace the defaults in `hard_detectors.py`.

**Reference:** Sajadi et al. (2022), *Nature Communications* 13, 2490.
https://doi.org/10.1038/s41467-022-30164-3

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

HEATMAP_DIR = Path('../data/heatmap_data/')
NOMINAL_FREQ = 60.0
ROCOF_THRESHOLD = 0.5   # Hz/s
FREQ_BAND = 0.5         # Hz


In [ ]:
# Load heatmap corpus (6 benchmarks x 1000 scenarios)
dfs = []
for f in sorted(HEATMAP_DIR.glob('*.csv')):
    df = pd.read_csv(f)
    df['source_file'] = f.stem
    dfs.append(df)

if dfs:
    data = pd.concat(dfs, ignore_index=True)
    print(f'Loaded {len(data)} scenarios from {len(dfs)} benchmark files.')
    print(data.columns.tolist())
else:
    print('No heatmap data found. Run simulations first or unzip heatmap_data.zip.')
    print('Generating synthetic calibration data for demonstration...')
    rng = np.random.default_rng(42)
    n = 6000
    data = pd.DataFrame({
        'rocof_hz_per_s':        rng.exponential(0.3, n),
        'freq_deviation_hz':     rng.exponential(0.2, n),
        'voltage_pu':            rng.uniform(0.88, 1.05, n),
        'thermal_margin_pct':    rng.uniform(0, 100, n),
        'reserve_margin_pct':    rng.uniform(0, 30, n),
        'inertia_coeff_min':     rng.uniform(0.05, 2.0, n),
        'headroom_reserve_pct':  rng.uniform(0, 25, n),
        'hertz_sec_score':       rng.exponential(0.05, n),
    })
    # Label: 1 = frequency excursion violated ROCOF/freq standard
    data['excursion_violated'] = (
        (data['rocof_hz_per_s'] > ROCOF_THRESHOLD) |
        (data['freq_deviation_hz'] > FREQ_BAND) |
        (data['reserve_margin_pct'] < 5.0) |
        (data['headroom_reserve_pct'] < 5.0)
    ).astype(int)


In [ ]:
# Build I_GEN feature matrix
features = {
    'w1_rocof_risk':               data['rocof_hz_per_s'].clip(0, ROCOF_THRESHOLD) / ROCOF_THRESHOLD,
    'w2_frequency_deviation':      data['freq_deviation_hz'].clip(0, FREQ_BAND) / FREQ_BAND,
    'w3_voltage_instability':      ((1.0 - data['voltage_pu']) / 0.10).clip(0, 1),
    'w4_thermal_violation':        (1.0 - data['thermal_margin_pct'] / 100.0).clip(0, 1),
    'w5_reserve_depletion':        (1.0 - data['reserve_margin_pct'] / 5.0).clip(0, 1),
    'w6_inertia_deficit':          (1.0 - data['inertia_coeff_min'] / 1.0).clip(0, 1),
    'w7_ramp_rate_excess':         data['rocof_hz_per_s'].clip(0, 0.10) / 0.10,
    'w8_power_quality':            (data['hertz_sec_score']).clip(0, 1),
    'w9_interconnection_stress':   (1.0 - data['thermal_margin_pct'] / 50.0).clip(0, 1),
    'w10_isolation_failure':       np.zeros(len(data)),  # binary; 0 in synthetic
    'w11_damping_failure':         (1.0 - data['headroom_reserve_pct'] / 20.0).clip(0, 1),
}
X = pd.DataFrame(features)
y = data['excursion_violated']
print(f'Feature matrix: {X.shape}, positive rate: {y.mean():.2%}')


In [ ]:
# Fit logistic regression to get calibrated weights
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
lr = LogisticRegression(max_iter=500, C=1.0)
cv_scores = cross_val_score(lr, X_scaled, y, cv=5, scoring='roc_auc')
print(f'Logistic Regression CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
lr.fit(X_scaled, y)

# Gradient boosting for comparison
gb = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)
gb_scores = cross_val_score(gb, X, y, cv=5, scoring='roc_auc')
print(f'Gradient Boosting CV AUC:    {gb_scores.mean():.3f} ± {gb_scores.std():.3f}')
gb.fit(X, y)


In [ ]:
# Extract and visualize calibrated weights
lr_weights = dict(zip(X.columns, np.abs(lr.coef_[0])))
lr_weights_norm = {k: v/sum(lr_weights.values()) for k, v in lr_weights.items()}

gb_weights = dict(zip(X.columns, gb.feature_importances_))

print('\nCalibrated I_GEN weights (Logistic Regression):')
for k, v in sorted(lr_weights_norm.items(), key=lambda x: -x[1]):
    print(f'  {k:<35} {v:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (weights, title) in zip(axes, [
    (lr_weights_norm, 'Logistic Regression (Normalized |coef|)'),
    (gb_weights, 'Gradient Boosting (Feature Importance)'),
]):
    sorted_items = sorted(weights.items(), key=lambda x: x[1], reverse=True)
    keys = [k.replace('w', 'w').replace('_', ' ') for k, _ in sorted_items]
    vals = [v for _, v in sorted_items]
    ax.barh(keys, vals, color='steelblue', edgecolor='black')
    ax.set_xlabel('Weight / Importance')
    ax.set_title(title)
    ax.invert_yaxis()
plt.suptitle('I_GEN Weight Calibration — Project Genesis ZetaScript', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('igen_weight_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: igen_weight_calibration.png')


In [ ]:
# Export calibrated weights as JSON for use in hard_detectors.py
import json
output = {
    'method': 'logistic_regression',
    'cv_auc_mean': float(cv_scores.mean()),
    'cv_auc_std': float(cv_scores.std()),
    'weights': {k.split('_', 1)[1]: round(v, 4) for k, v in lr_weights_norm.items()},
    'note': 'Replace WEIGHTS dict in hard_detectors.py with these values.',
}
with open('calibrated_igen_weights.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Saved: calibrated_igen_weights.json')
print(json.dumps(output, indent=2))
